# Hierarchical Reasoning Model (2025)

[[paper]](https://arxiv.org/abs/2506.21734)

Решение от молодого стартапа Sapient Intelligence, попытка сделать reasoning на внутренних состояних сети, а не с помощью текстовой генерации (Chain-Of-Though), как это в основном делается в "думающих" LLM моделях

*Reasoninig = процесс решения задач, требующий сложного многошагового планирования действий

__Мотивация:__ текущие решения слишком "плоские", в смысле, что реализуют только forward pass фиксированной глубины, что затрудняет их возможности по решению задач. 

Это даже теоретически обосновывается - текущие архитектуры сетей не Тьюринг-полные (не способны решать задачи любой сожности). По классификации сложности схем они относятся к классам $AC_0$ и $TC_0$. $AC_0$ - булевы функции фиксированной глубины, $TC_0$ - то же что $AC_0$, но с пороговыой функцией

__Идея:__ сделать рекуррентную RNN сеть, но использовать две отдельные магистрали передачи сигнала - внешнюю (H-модуль, high level) и внутреннюю (L-module, low level), выполнение которых чередуется. Внешняя будет отвечать за верхнеуровневую формулировку текущей подзадачи (что-то типа планировщика или оценщика), а внутренняя за её решение. Сигнал от внешней магистрали подается на вход внутренней

Авторы многократно акцентируют внимание, что вдохновлено биологическими паттернами процессов размышления в мозге. Но это скорее всего, стандартное притягивание за уши для маркетинга

<img src="img/hrm1.png" width=350>

Авторы также проводят параллель с популяризированными Канеманом системе 1 (быстрому интуитивному восприятию) и системе 2 (медленному осмысленному размышлению) - двумя режимами работы мозга, которые, как утверждается, чередуются при решении задач

В статье модель описывается как общий подход, без акцента на рещаемую задачу. Важно, что акцент не на генеративные модели. Там в качестве примеров скорее BERT подобные задачи. Вход статичный, читается один раз один раз и это просто какой-то вектор.

Алгоритм (Inference):
- Внешний H слой возвращает состояние $h_m$<br>
- Это состояние играет роль условия - оно объединяется с вектором эмбедингов и T раз прогоняется через L-модуль<br>
- L-моддуль - это небольшой трансофрмер<br>
- Получаем выходной эмбединг и сжимаем его в основной результат - эмбединг [CLS] токена<br>

<img src="img/hrm3.png" width=550>

Кол-во внешних циклов $N$ адаптивно - модель решает, запускать еще цикл или нет, а кол-во внутренних $T$ фиксировано, применяется ровно T вызовов функции циклов

Hierarchical Convergence. 

## Training

Процесс обучения называется __Deep Supervision__.

__Inference Scaling__ - динамическое определение продолжительности (кол-ва циклов) размышления. Оно полезно, когда время размышления влияет на качество ответа - на простых задачах можно сэкономить, а на сложных улучшить качество ответа. 

В HRM заимствуют известный еще с 2016 года подход __Adaptive Computation Time ([ACT](https://arxiv.org/pdf/1603.08983), 2016)__. Это способ динамического определения момента времени в рекуррентных сетях, когда пришли к ответу. В HRM он реализуется для внешнего H-цикла. Кроме того Inference Scaling регулируется через установку лимита на максимальное кол-во циклов.

После завершения каждого внешнего цикла вычисляется дополнительный выход Q-head, который вероятность того, что полученый от внутренней сети результат $y$ и внутреннее состояние $z$ дают правильный ответ. Q-head представляет собой простой линейный слой, который дает два логита $Q_{halt}$ и $Q_{cont}$, активированные softmax-ом

Если $Q_{halt} > Q_{cont}$, стопаем расчет и считаем результат $y$ финальным. Порог не обязательно 0.5, он зависит от выбранного баланса точности и экономии. При этом также проверяем, что выполнилось минимальное кол-во циклов $Mmin$ и не превышено максимальное $Mmax$



One-step gradient approximation.

Обучается этот Q-лосс вместе с основным лоссом $$L = L(\hat{y},y) + \text{BCE}(Q_{halt},y)$$

Почему подход успешный? Очень эффективаня модель: 
- всего 27M параметров (за счет того, что сеть рекурентная параметры повторяются)
- при этом выигрывает на сложных задачах даже у ChatGPT (но проигрывает на стандартных)

<img src="img/hrm2.png" width=500>


